<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/FINALNY_NOTATNIK_PORTFOLIO_Klasyfikacja_cukrzycy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Przesłane pliki:", list(uploaded.keys()))

# End-to-End Projekt Klasyfikacji: Diagnoza Cukrzycy

**Autor:** [Tomasz Wienke]  
**Cel biznesowy:** Zbudowanie modelu uczenia maszynowego, który na podstawie danych diagnostycznych pomoże przewidzieć, czy pacjent ma cukrzycę.  
**Kluczowe metryki:** `F1-score` i `AUC` (ze względu na niezbalansowanie klas i wysoki koszt przeoczenia chorego).  
**Wnioski:** Model AdaBoost po strojeniu osiągnął F1=0.92, przewyższając znacząco model bazowy.

**Umiejętności:** Python, Pandas, Seaborn, Scikit-learn, EDA, Regresja Logistyczna, SVM, Random Forest, AdaBoost, GridSearchCV.

## 1. Przygotowanie środowiska i wczytanie danych

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Modele
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# Wczytanie
diabetes = pd.read_csv('diabetes.csv')

# Cechy i cel
features = ['Pregnancies', 'PlasmaGlucose', 'DiastolicBloodPressure',
            'TricepsThickness', 'SerumInsulin', 'BMI', 'DiabetesPedigree', 'Age']
target = 'Diabetic'

X = diabetes[features]
y = diabetes[target]

## 2. Eksploracyjna Analiza Danych (EDA)

In [ ]:
# Balans klas
print("Balans klas:")
print(y.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

sns.countplot(x=y)
plt.title('Rozkład klas (0 – Zdrowy, 1 – Chory)')
plt.show()

# Średnie wartości cech w podziale na klasy
print("\nŚrednie wartości cech:")
display(X.groupby(y).mean())

# Przykładowe rozkłady
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for i, col in enumerate(features):
    ax = axes[i//4, i%4]
    sns.histplot(data=diabetes, x=col, hue=target, kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 3. Podział danych (Trening / Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)
print(f'Trening: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4. Model bazowy – Regresja Logistyczna (punkt odniesienia)

In [ ]:
base_model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42, max_iter=2000))
base_model.fit(X_train, y_train)
y_pred_base = base_model.predict(X_test)

print("Model bazowy (Regresja Logistyczna):")
print(classification_report(y_test, y_pred_base))
f1_base = f1_score(y_test, y_pred_base)
print(f'F1-score bazowy: {f1_base:.4f}')

## 5. Strojenie hiperparametrów (GridSearchCV)

Wybieramy najlepsze hiperparametry dla wszystkich kluczowych modeli. Wykorzystujemy `f1_macro`, aby zbalansować skuteczność dla obu klas.

In [ ]:
models = [
    {
        'name': 'RandomForest',
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]}
    },
    {
        'name': 'AdaBoost',
        'estimator': AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), random_state=42),
        'params': {'n_estimators': [50, 100, 200]}
    },
    {
        'name': 'SVM',
        'estimator': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
        'params': {'svc__kernel': ['rbf', 'linear'], 'svc__C': [0.1, 1, 10]}
    },
    {
        'name': 'KNN',
        'estimator': make_pipeline(StandardScaler(), KNeighborsClassifier()),
        'params': {'kneighborsclassifier__n_neighbors': [3, 5, 9, 15], 'kneighborsclassifier__metric': ['euclidean']}
    }
]

In [ ]:
final_results = [{'Model': 'LogisticRegression (Bazowy)', 'F1_Test': f1_base, 'AUC': roc_auc_score(y_test, y_pred_base)}]

for model_info in models:
    print(f'\n--- Strojenie: {model_info["name"]} ---')
    grid = GridSearchCV(model_info['estimator'], model_info['params'],
                        scoring='f1_macro', cv=5, verbose=0, n_jobs=-1)
    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    y_pred = best.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])

    final_results.append({'Model': model_info['name'], 'F1_Test': f1, 'AUC': auc})
    print(f'{model_info["name"]}: F1={f1:.4f}, AUC={auc:.4f}')

## 6. Podsumowanie i wnioski końcowe

In [ ]:
df_final = pd.DataFrame(final_results).sort_values('F1_Test', ascending=False)

print("=== TABELA KOŃCOWA ===")
print(df_final.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.bar(df_final['Model'], df_final['F1_Test'], color='skyblue')
plt.title('Porównanie F1-score wszystkich modeli')
plt.ylabel('F1-score')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 7. Wnioski

1.  **Zwycięzca – AdaBoost (F1 = 0.924, AUC = 0.989):**
    Model ten okazał się najlepszy, minimalnie wyprzedzając Random Forest. Jego sekwencyjna natura (skupianie się na trudnych przypadkach) świetnie sprawdziła się w tym problemie medycznym.

2.  **Random Forest na podium (F1 = 0.911):**
    Las losowy potwierdził, że jest niezawodnym, stabilnym wyborem. Jego przewagą jest odporność na szum i łatwość użycia.

3.  **SVM trzyma poziom (F1 = 0.843):**
    Jest solidnym modelem, ale jego wyniki są nieco niższe od liderów. Wymaga też starannego skalowania i jest wolniejszy przy dużych zbiorach.

4.  **KNN nie polubił się z wieloma cechami (F1 = 0.793):**
    To największe zaskoczenie. Przy 2 cechach KNN był blisko podium. Przy 8 cechach jego skuteczność spadła. To klasyczny objaw **przekleństwa wymiarowości** – przy dużej liczbie wymiarów odległości między punktami stają się mniej miarodajne, a KNN traci swoją moc.

5.  **Model bazowy – punkt startowy (F1 = 0.664):**
    Regresja logistyczna z domyślnymi parametrami jest najsłabsza, ale stanowiła kluczowy punkt odniesienia. Bez niej nie wiedzielibyśmy, jak duży postęp zrobiliśmy.

### 🚀 Rekomendacja biznesowa

Do wdrożenia w aplikacji diagnostycznej rekomenduję model **AdaBoost** (lub Random Forest jako solidną alternatywę). Osiągnięte wyniki (Recall dla klasy chorych = 0.93) oznaczają, że model wykrywa 93% pacjentów z cukrzycą, co jest bardzo dobrym wynikiem klinicznym.